In [ ]:
#Distribuição da populção 15_24 pelos edifícios

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from shapely.wkt import loads

# Carregar os dados
bgri = gpd.read_file("BGRI2021_1312.gpkg")
edificios = pd.read_csv("edificios.csv")

# Converter 'edificios' para GeoDataFrame, garantindo que a coluna de geometria está correta
edificios['geometry'] = edificios['geometry'].apply(loads)
edificios = gpd.GeoDataFrame(edificios, geometry='geometry')

# Definir o CRS e converter para EPSG:3763
edificios.set_crs(epsg=4326, inplace=True)
edificios = edificios.to_crs(epsg=3763)
bgri = bgri.to_crs(epsg=3763)

# Remover a coluna 'index_right' se existir
if 'index_right' in edificios.columns:
    edificios.drop(columns=['index_right'], inplace=True)
if 'index_right' in bgri.columns:
    bgri.drop(columns=['index_right'], inplace=True)

# Corrigir possíveis erros topológicos nas geometrias
edificios['geometry'] = edificios['geometry'].buffer(0)
bgri['geometry'] = bgri['geometry'].buffer(0)

# Realizar a junção espacial para associar cada edifício a uma subseção estatística
edificios = gpd.sjoin(edificios, bgri[['SUBSECCAO', 'geometry']], how='left', predicate='intersects')

# Mesclar os dados dos edifícios com os dados de população do BGRI para o grupo 15-24
edificios = edificios.merge(
    bgri[['SUBSECCAO', 'N_INDIVIDUOS_15_24']],
    on='SUBSECCAO',
    how='left'
)

# Calcular a área de cada edifício e o total de área por subseção
edificios['area'] = edificios['geometry'].area
soma_areas = edificios.groupby('SUBSECCAO')['area'].sum().reset_index()
soma_areas.rename(columns={'area': 'TOTAL_AREA_SE'}, inplace=True)
edificios = edificios.merge(soma_areas, on='SUBSECCAO', how='left')

# --- Distribuição para o grupo 15-24 ---
edificios['POP_15_24_PROPORCIONAL'] = (
    (edificios['area'] / edificios['TOTAL_AREA_SE']) * edificios['N_INDIVIDUOS_15_24']
)
edificios['POP_15_24_ARREDONDADA'] = edificios['POP_15_24_PROPORCIONAL'].round()

# Ajuste para que a soma dos valores arredondados por subseção seja igual a N_INDIVIDUOS_15_24
ajustes_15_24 = (
    edificios.groupby('SUBSECCAO')['POP_15_24_ARREDONDADA'].sum() -
    edificios.groupby('SUBSECCAO')['N_INDIVIDUOS_15_24'].first()
).reset_index()
ajustes_15_24.rename(columns={0: 'AJUSTE'}, inplace=True)

# Ajustar a população arredondada de maneira mais segura e distribuída, sem exceder a população da subseção
for _, row in ajustes_15_24.iterrows():
    subsecao = row['SUBSECCAO']
    diff = int(row['AJUSTE'])
    
    if diff == 0:
        continue

    subset = edificios[edificios['SUBSECCAO'] == subsecao]
    if subset.empty:
        continue
    
    # Ordenar os edifícios por população estimada
    subset_sorted = subset.sort_values(by='POP_15_24_ARREDONDADA', ascending=(diff > 0))
    
    # Garantir que a soma não ultrapasse a população da subseção
    total_subseccao = subset['N_INDIVIDUOS_15_24'].iloc[0]  # Total da subseção
    current_total = subset_sorted['POP_15_24_ARREDONDADA'].sum()
    max_possible_diff = total_subseccao - current_total
    
    # Se a diferença for maior que o que pode ser ajustado sem exceder o total, usamos o valor máximo possível
    diff = int(min(diff, max_possible_diff))
    
    # Distribuir o ajuste de forma cíclica, evitando que valores fiquem negativos
    for i in range(abs(diff)):
        idx = subset_sorted.index[i % len(subset_sorted)]
        if diff > 0:
            # Reduzir a população, mas garantindo que não fique negativa
            if edificios.loc[idx, 'POP_15_24_ARREDONDADA'] > 0:
                edificios.loc[idx, 'POP_15_24_ARREDONDADA'] -= 1
        else:
            # Aumentar a população (caso haja déficit)
            if edificios.loc[idx, 'POP_15_24_ARREDONDADA'] < total_subseccao:
                edificios.loc[idx, 'POP_15_24_ARREDONDADA'] += 1

# --- Bloco de ajuste final por SUBSECCAO ---
# Aqui garantimos que, para cada SUBSECCAO, a soma dos valores não exceda o total oficial
for sub in edificios['SUBSECCAO'].unique():
    subset = edificios[edificios['SUBSECCAO'] == sub]
    official = subset['N_INDIVIDUOS_15_24'].iloc[0]
    current = subset['POP_15_24_ARREDONDADA'].sum()
    diff_final = int(current - official)
    if diff_final > 0:
        # Se há excesso, subtrair dif_final distribuindo entre os edifícios com maiores valores
        subset_sorted = subset.sort_values(by='POP_15_24_ARREDONDADA', ascending=False)
        for i in range(diff_final):
            idx = subset_sorted.index[i % len(subset_sorted)]
            if edificios.loc[idx, 'POP_15_24_ARREDONDADA'] > 0:
                edificios.loc[idx, 'POP_15_24_ARREDONDADA'] -= 1
    elif diff_final < 0:
        # Se há déficit, distribuir adicionando indivíduos (se permitido)
        subset_sorted = subset.sort_values(by='POP_15_24_ARREDONDADA', ascending=True)
        for i in range(abs(diff_final)):
            idx = subset_sorted.index[i % len(subset_sorted)]
            edificios.loc[idx, 'POP_15_24_ARREDONDADA'] += 1

# Garantir que não haja valores negativos após os ajustes
edificios['POP_15_24_ARREDONDADA'] = edificios['POP_15_24_ARREDONDADA'].clip(lower=0)

# Converter as coordenadas para EPSG:4326 para visualização no Folium
edificios = edificios.to_crs(epsg=4326)
bgri = bgri.to_crs(epsg=4326)

# Calcular a população total 15-24 por SUBSECCAO para visualização
pop_bgri = (
    edificios.groupby('SUBSECCAO')['POP_15_24_ARREDONDADA']
    .sum()
    .reset_index()
    .rename(columns={'POP_15_24_ARREDONDADA': 'POP_TOTAL_15_24'})
)
bgri = bgri.merge(pop_bgri, on='SUBSECCAO', how='left')
bgri['POP_TOTAL_15_24'] = bgri['POP_TOTAL_15_24'].fillna(0)

# Definir o centro do mapa (aproximadamente o centro da cidade do Porto)
centro = [41.14961, -8.61099]

# Criar um mapa Folium centrado no Porto
m = folium.Map(location=centro, zoom_start=14)

# Adicionar os polígonos dos edifícios ao mapa, colorindo conforme a população 15-24
for _, row in edificios.iterrows():
    color = 'red' if row['POP_15_24_ARREDONDADA'] > 10 else 'blue'
    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': color,
            'weight': 1,
            'fillOpacity': 0.6
        },
        tooltip=(
            f"Edifício {row['osm_id']}<br>"
            f"População 15-24: {row['POP_15_24_ARREDONDADA']}"
        )
    ).add_to(m)

# Adicionar os polígonos das subseções do BGRI ao mapa com transparência e tooltips com a população total 15-24
folium.GeoJson(
    bgri,
    style_function=lambda x: {
        'fillColor': 'green',
        'color': 'green',
        'weight': 1,
        'fillOpacity': 0.1
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['POP_TOTAL_15_24'], 
        aliases=['População Total 15-24']
    )
).add_to(m)

# Verificar a soma da população ajustada
pop_total_adjusted = edificios['POP_15_24_ARREDONDADA'].sum()
pop_total_bagri = bgri['N_INDIVIDUOS_15_24'].sum()

print(f"População total ajustada: {pop_total_adjusted}")
print(f"População total BGRI: {pop_total_bagri}") 

# Exibir o mapa no Notebook
display(m)

# Salvar o resultado em um arquivo CSV
edificios[['osm_id', 'SUBSECCAO', 'area', 'POP_15_24_ARREDONDADA']].to_csv(
    'pop_reconstruido_inteiro.csv',
    index=False
)

print("Processo concluído. Ficheiro salvo como pop15_24.csv")


In [ ]:
#Distribuição da populção 65 anos ou + pelos edifícios

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from shapely.wkt import loads, dumps  # Importar a função dumps

# Carregar os dados
bgri = gpd.read_file("BGRI2021_1312.gpkg")
edificios = pd.read_csv("edificios.csv")

# Converter 'edificios' para GeoDataFrame, garantindo que a coluna de geometria está correta
edificios['geometry'] = edificios['geometry'].apply(loads)
edificios = gpd.GeoDataFrame(edificios, geometry='geometry')

# Definir o CRS e converter para EPSG:3763
edificios.set_crs(epsg=4326, inplace=True)
edificios = edificios.to_crs(epsg=3763)
bgri = bgri.to_crs(epsg=3763)

# Remover a coluna 'index_right' se existir
if 'index_right' in edificios.columns:
    edificios.drop(columns=['index_right'], inplace=True)
if 'index_right' in bgri.columns:
    bgri.drop(columns=['index_right'], inplace=True)

# Corrigir possíveis erros topológicos nas geometrias
edificios['geometry'] = edificios['geometry'].buffer(0)
bgri['geometry'] = bgri['geometry'].buffer(0)

# Realizar a junção espacial para associar cada edifício a uma subseção estatística
edificios = gpd.sjoin(edificios, bgri[['SUBSECCAO', 'geometry']], how='left', predicate='intersects')

# Mesclar os dados dos edifícios com os dados de população do BGRI para o grupo 65 ou mais
edificios = edificios.merge(
    bgri[['SUBSECCAO', 'N_INDIVIDUOS_65_OU_MAIS']],
    on='SUBSECCAO',
    how='left'
)

# Calcular a área de cada edifício e o total de área por subseção
edificios['area'] = edificios['geometry'].area
soma_areas = edificios.groupby('SUBSECCAO')['area'].sum().reset_index()
soma_areas.rename(columns={'area': 'TOTAL_AREA_SE'}, inplace=True)
edificios = edificios.merge(soma_areas, on='SUBSECCAO', how='left')

# --- Distribuição para o grupo 65 ou mais ---
edificios['POP_65_OU_MAIS_PROPORCIONAL'] = (
    (edificios['area'] / edificios['TOTAL_AREA_SE']) * edificios['N_INDIVIDUOS_65_OU_MAIS']
)
edificios['POP_65_OU_MAIS_ARREDONDADA'] = edificios['POP_65_OU_MAIS_PROPORCIONAL'].round()

# Ajuste para que a soma dos valores arredondados por subseção seja igual a N_INDIVIDUOS_65_OU_MAIS
ajustes_65_ou_mais = (
    edificios.groupby('SUBSECCAO')['POP_65_OU_MAIS_ARREDONDADA'].sum() -
    edificios.groupby('SUBSECCAO')['N_INDIVIDUOS_65_OU_MAIS'].first()
).reset_index()
ajustes_65_ou_mais.rename(columns={0: 'AJUSTE'}, inplace=True)

# Ajustar a população arredondada de maneira mais segura e distribuída, sem exceder a população da subseção
for _, row in ajustes_65_ou_mais.iterrows():
    subsecao = row['SUBSECCAO']
    diff = int(row['AJUSTE'])
    
    if diff == 0:
        continue

    subset = edificios[edificios['SUBSECCAO'] == subsecao]
    if subset.empty:
        continue
    
    # Ordenar os edifícios por população estimada
    subset_sorted = subset.sort_values(by='POP_65_OU_MAIS_ARREDONDADA', ascending=(diff > 0))
    
    # Garantir que a soma não ultrapasse a população da subseção
    total_subseccao = subset['N_INDIVIDUOS_65_OU_MAIS'].iloc[0]  # Total da subseção
    current_total = subset_sorted['POP_65_OU_MAIS_ARREDONDADA'].sum()
    max_possible_diff = total_subseccao - current_total
    
    # Se a diferença for maior que o que pode ser ajustado sem exceder o total, usamos o valor máximo possível
    diff = int(min(diff, max_possible_diff))
    
    # Distribuir o ajuste de forma cíclica, evitando que valores fiquem negativos
    for i in range(abs(diff)):
        idx = subset_sorted.index[i % len(subset_sorted)]
        if diff > 0:
            # Reduzir a população, mas garantindo que não fique negativa
            if edificios.loc[idx, 'POP_65_OU_MAIS_ARREDONDADA'] > 0:
                edificios.loc[idx, 'POP_65_OU_MAIS_ARREDONDADA'] -= 1
        else:
            # Aumentar a população (caso haja déficit)
            if edificios.loc[idx, 'POP_65_OU_MAIS_ARREDONDADA'] < total_subseccao:
                edificios.loc[idx, 'POP_65_OU_MAIS_ARREDONDADA'] += 1

# --- Bloco de ajuste final por SUBSECCAO ---
# Aqui garantimos que, para cada SUBSECCAO, a soma dos valores não exceda o total oficial
for sub in edificios['SUBSECCAO'].unique():
    subset = edificios[edificios['SUBSECCAO'] == sub]
    official = subset['N_INDIVIDUOS_65_OU_MAIS'].iloc[0]
    current = subset['POP_65_OU_MAIS_ARREDONDADA'].sum()
    diff_final = int(current - official)
    if diff_final > 0:
        # Se há excesso, subtrair dif_final distribuindo entre os edifícios com maiores valores
        subset_sorted = subset.sort_values(by='POP_65_OU_MAIS_ARREDONDADA', ascending=False)
        for i in range(diff_final):
            idx = subset_sorted.index[i % len(subset_sorted)]
            if edificios.loc[idx, 'POP_65_OU_MAIS_ARREDONDADA'] > 0:
                edificios.loc[idx, 'POP_65_OU_MAIS_ARREDONDADA'] -= 1
    elif diff_final < 0:
        # Se há déficit, distribuir adicionando indivíduos (se permitido)
        subset_sorted = subset.sort_values(by='POP_65_OU_MAIS_ARREDONDADA', ascending=True)
        for i in range(abs(diff_final)):
            idx = subset_sorted.index[i % len(subset_sorted)]
            edificios.loc[idx, 'POP_65_OU_MAIS_ARREDONDADA'] += 1

# Garantir que não haja valores negativos após os ajustes
edificios['POP_65_OU_MAIS_ARREDONDADA'] = edificios['POP_65_OU_MAIS_ARREDONDADA'].clip(lower=0)

# Converter as coordenadas para EPSG:4326 para visualização no Folium
edificios = edificios.to_crs(epsg=4326)
bgri = bgri.to_crs(epsg=4326)

# Calcular a população total 65 ou mais por SUBSECCAO para visualização
pop_bgri = (
    edificios.groupby('SUBSECCAO')['POP_65_OU_MAIS_ARREDONDADA']
    .sum()
    .reset_index()
    .rename(columns={'POP_65_OU_MAIS_ARREDONDADA': 'POP_TOTAL_65_OU_MAIS'})
)
bgri = bgri.merge(pop_bgri, on='SUBSECCAO', how='left')
bgri['POP_TOTAL_65_OU_MAIS'] = bgri['POP_TOTAL_65_OU_MAIS'].fillna(0)

# Definir o centro do mapa (aproximadamente o centro da cidade do Porto)
centro = [41.14961, -8.61099]

# Criar um mapa Folium centrado no Porto
m = folium.Map(location=centro, zoom_start=14)

# Adicionar os polígonos dos edifícios ao mapa, colorindo conforme a população 65+
for _, row in edificios.iterrows():
    color = 'purple' if row['POP_65_OU_MAIS_ARREDONDADA'] > 10 else 'blue'
    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': color,
            'weight': 1,
            'fillOpacity': 0.6
        },
        tooltip=(
            f"Edifício {row['osm_id']}<br>"
            f"População 65+: {row['POP_65_OU_MAIS_ARREDONDADA']}"
        )
    ).add_to(m)

# Adicionar os polígonos das subseções do BGRI ao mapa com transparência e tooltips com a população total 65+
folium.GeoJson(
    bgri,
    style_function=lambda x: {
        'fillColor': 'green',
        'color': 'green',
        'weight': 1,
        'fillOpacity': 0.1
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['POP_TOTAL_65_OU_MAIS'], 
        aliases=['População Total 65+']
    )
).add_to(m)

# Verificar a soma da população ajustada
pop_total_adjusted = edificios['POP_65_OU_MAIS_ARREDONDADA'].sum()
pop_total_bagri = bgri['N_INDIVIDUOS_65_OU_MAIS'].sum()

print(f"População total ajustada 65+: {pop_total_adjusted}")
print(f"População total BGRI 65+: {pop_total_bagri}") 

# Exibir o mapa no Notebook
display(m)

# Salvar o resultado em um arquivo CSV com a geometria convertida para WKT
edificios['geometry_wkt'] = edificios['geometry'].apply(dumps)  # Converter geometria para WKT
edificios[['osm_id', 'SUBSECCAO', 'area', 'POP_65_OU_MAIS_ARREDONDADA', 'geometry_wkt']].to_csv(
    'pop_reconstruido_65_ou_mais.csv',
    index=False
)

print("Processo concluído. Ficheiro salvo como pop_reconstruido_65_ou_mais.csv")

In [ ]:
🏙️ Population Distribution in Buildings Based on BGRI Data

This repository contains scripts and data files for estimating and visualizing the elderly population (65+) distribution in buildings based on BGRI statistical subsections in Porto, Portugal. The project utilizes Python, GeoPandas, Folium, and OSMnx for spatial analysis and visualization.
📌 Project Overview

The goal of this project is to:

    Analyze the spatial distribution of the elderly population (65+) in buildings.
    Compute population estimates per building based on BGRI statistical data.
    Distribute the population proportionally to building areas.
    Generate an interactive Folium map showing population density per building.

📂 Files in this Repository
File 	Description
BGRI2021_1312.gpkg 	GeoPackage containing BGRI statistical zones for Porto.
edificios.csv 	CSV file with building locations and geometries (WKT format).
population_distribution.ipynb 	Jupyter Notebook calculating the 65+ population distribution in buildings.
🛠️ Setup & Dependencies

To run the Jupyter notebooks, install the necessary dependencies:

pip install pandas geopandas osmnx folium networkx shapely joblib matplotlib numpy

